## This notebook intends to address task1 -- Capacity Plan
the task1 has two parts:

1. Calcualte hoursly demand prediction for each parts via breaking down annual pull demand from client A, B into 
2. Transform demand for production into capacity requirements for equipments, personnel via formulating and solving an IP problem for best set up of resources.

__Note that the planning of resources can be changed in later tasks if for the need from layout design__

In [14]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus

### Demand Calculation

the information on annual demand turns out to be redundant only set to help conceptualize the scenario

In [15]:
# parameter settings:

products = ['A1', 'A2', 'A3', 'B1', 'B2']

# weekly demand 
demand_data = {
    'Weekly_Mean': [
        962,  
        1923, 
        2500, 
        1154, 
        1538  
    ],
    'Weekly_CV': [0.15, 0.2, 0.2, 0.12, 0.18] 
}

demand_df = pd.DataFrame(demand_data, index=products) 

# the BOM of requirements of each Products (A, B) from each Parts (P1~P20)
bom_df = pd.read_csv('../Data/BOM.csv', index_col=0).T # for the later matrix multiplication dim align
print('BOM of product to parts:\n', bom_df.head())


BOM of product to parts:
     A1  A2  A3  B1  B2
P1   1   2   4   4   1
P2   4   0   2   2   0
P3   0   0   0   4   1
P4   0   1   2   2   0
P5   0   1   2   0   0


since clients require 99.5% of service level for replenishment from FaMoSa, the target capacity is set to be 99.5% percentile of the demand

there is limited information on familty of distribution for demand, with following justifications:
- the demand sample has a moderately large size ($\mu > 30$)
- the variation is mild ($\sigma < \mu$)


thus we assume the demand distribution follows normal

In [16]:
# calculate the target capacity for products
Z_score = norm.ppf(0.995)

demand_df['Weekly_std'] = demand_df['Weekly_Mean'] * demand_df['Weekly_CV']
# round up to maintain integer
demand_df['Product_Capacity'] = np.ceil(demand_df['Weekly_Mean'] + demand_df['Weekly_std']*Z_score)


# Aggregate to parts demand
part_demand = pd.DataFrame()
part_demand['Part_Weekly_Capacity'] = bom_df.dot(demand_df['Product_Capacity'])

part_demand['Part_Weekly_Capacity'] = np.ceil(part_demand['Part_Weekly_Capacity']) 

# Fix: Strip whitespace from the index
part_demand.index = part_demand.index.str.strip()

print('weekly capacity for each product:\n', demand_df['Product_Capacity'])
print('\nweekly capacity for each part:\n', part_demand)

weekly capacity for each product:
 A1    1334.0
A2    2914.0
A3    3788.0
B1    1511.0
B2    2252.0
Name: Product_Capacity, dtype: float64

weekly capacity for each part:
      Part_Weekly_Capacity
P1                30610.0
P2                15934.0
P3                 8296.0
P4                13512.0
P5                10490.0
P6                 5838.0
P7                20664.0
P8                 6044.0
P9                13404.0
P10               18289.0
P11               11676.0
P12               14344.0
P13                7172.0
P14               26758.0
P15                3022.0
P16               18066.0
P17                7172.0
P18               17258.0
P19               29276.0
P20               21725.0


## Capacity Planning

after validating the demand for each part, in this case each parts are produced through a series of processes naming A, B, C...M. Meanwhile, some machine can do one step and some can do multiple and that is showcased via their naming (e.g. machien AB can do A and B step)

so the steps to do the planning:
1. calculate working hour requirements for each step 
$$D_s = \sum_{P_i} D_{P_i} t_{{P_i}s}$$
in which 
- $D_{P_i}$ for demand of product $P_i$, 
- $t_{{P_i}X}$ for time required to finish step $X$ of product $P_i$
 

In [17]:
# load parameters

path = "../Data"  # Go up one level then into Data
step = pd.read_csv(f"{path}/Steps.csv", index_col=0)
timecost = pd.read_csv(f"{path}/Timecost.csv", index_col=0)
 
print(step.head())
print("\n\n", timecost.head())

     Step 1 Step 2 Step 3 Step 4 Step 5 Step 6 Step 7
Part                                                 
P1        B      A      B      C      D      I      J
P2        A      C      D      H      J    NaN    NaN
P3        B      D      C      I      J    NaN    NaN
P4        A      B      D      G      H    NaN    NaN
P5        B      C      C      I    NaN    NaN    NaN


       Step 1  Step 2  Step 3  Step 4  Step 5  Step 6  Step 7
Part                                                        
P1      2.50    1.00    2.50    0.50    2.50    1.25     2.5
P2      1.25    0.50    2.50    1.00    2.50     NaN     NaN
P3      1.75    3.00    0.75    1.50    2.50     NaN     NaN
P4      1.00    2.00    3.00    0.25    1.25     NaN     NaN
P5      1.50    0.75    3.50    1.75     NaN     NaN     NaN


In [18]:
# Check the structure of your DataFrames
print("Timecost index:", timecost.index)
print("Part_demand keys:", part_demand.keys())
print("Part_demand index:", part_demand.index)
# the output for the part_demand index used to have a space before P1 (" P1"), thus when trying to access P1 index row of this df,
# it returns keyError

Timecost index: Index(['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10', 'P11',
       'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20'],
      dtype='object', name='Part')
Part_demand keys: Index(['Part_Weekly_Capacity'], dtype='object')
Part_demand index: Index(['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10', 'P11',
       'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20'],
      dtype='object')


In [19]:
# calculate the working hour requirements


time_matrix = pd.DataFrame(index=timecost.index, columns=timecost.columns)


for part in timecost.index: # use label indexing but can also use integer iloc
    time_matrix.loc[part] = timecost.loc[part] * part_demand.loc[part, 'Part_Weekly_Capacity']


step_total_time = {}

# Get unique steps (A, B, C, etc.) excluding NaN
unique_steps = step.values.ravel()
unique_steps = pd.unique([x for x in unique_steps if pd.notna(x)])

# Calculate total time for each step type
for step_type in unique_steps:
    # boolean mask
    mask = (step == step_type)
    # Sum the corresponding times in time_matrix where mask is True
    total_time = (time_matrix * mask).sum().sum()
    step_total_time[step_type] = total_time


step_summary = pd.DataFrame({
    'Step': list(step_total_time.keys()),
    'Total_Time': list(step_total_time.values())
})

print("Summary of time requirements by step type:\n", step_summary)

Summary of time requirements by step type:
    Step  Total_Time
0     B   217624.50
1     A    68418.00
2     C    92493.50
3     D   199298.00
4     I   261138.25
5     J   325629.50
6     H   209805.50
7     G   145888.00
8     E   153931.00
9     F   144810.75
10    K    43245.25
11    L   210961.50
12    M   337969.50


/tmp/ipykernel_31500/780355344.py:15: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  unique_steps = pd.unique([x for x in unique_steps if pd.notna(x)])
